# Quantum Harmonic Oscillator

An interactive HTML5 simulation of a quantum harmonic oscillator. The original simulation is generated as HTML and embedded directly in the notebook, so it does not depend on legacy Jupyter widget state.

In [ ]:
from IPython.display import HTML, display

html_template = r'''
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
body { font-family: Arial, sans-serif; }
.wrap { width: 100%; max-width: 760px; margin: auto; text-align: center; }
canvas { border: 1px solid #aaa; max-width: 100%; height: auto; background: black; }
.row { margin: 8px 0; }
button { padding: 5px 12px; margin: 2px; cursor: pointer; }
label { margin: 0 5px; }
.notes { text-align: left; margin-top: 12px; }
</style>
</head>
<body>
<div class="wrap">
<h2>Quantum Harmonic Oscillator</h2>
<canvas id="theCanvas" width="600" height="300"></canvas>
<div class="row">
<button id="pauseButton">Pause</button>
Speed: <input type="range" id="speedSlider" min="0" max="0.2" step="0.001" value="0.05">
<label><input type="radio" name="plotType" id="realImag"> Real/imag</label>
<label><input type="radio" name="plotType" checked> Density/phase</label>
</div>
<div class="row">
<button id="zeroButton">Zero</button>
<button id="normalizeButton">Normalize</button>
<button id="coherentButton">Coherent(α)</button>
α = <span id="alphaReadout">1.0</span>
<input type="range" id="alphaSlider" min="0" max="4" step="0.1" value="1">
</div>
<div class="notes">
<p>Circles display phasor diagrams for the complex amplitudes of the basis functions, with the ground state on the left.</p>
<p>The wavefunction is constructed by adding eight basis functions multiplied by their corresponding complex amplitudes.</p>
<p>Over time, each basis amplitude rotates in the complex plane at a frequency proportional to its energy.</p>
<p>Real and imaginary parts are shown in orange and blue. Probability density and phase are shown in density/phase mode.</p>
<p>Click and drag the lines on the clock faces to adjust the corresponding amplitude and phase.</p>
<p>A truncated approximation to a coherent state can be created with the α control.</p>
</div>
</div>
<script>
const canvas = document.getElementById('theCanvas');
const ctx = canvas.getContext('2d');
const pauseButton = document.getElementById('pauseButton');
const speedSlider = document.getElementById('speedSlider');
const realImag = document.getElementById('realImag');
const alphaSlider = document.getElementById('alphaSlider');
const alphaReadout = document.getElementById('alphaReadout');
const iMax = canvas.width;
const clockSpaceFraction = 0.25;
const clockRadiusFraction = 0.45;
const nMax = 7;
const nColors = 360;
const psi = {re: new Array(iMax + 1), im: new Array(iMax + 1)};
const eigenPsi = new Array(nMax + 1);
const amplitude = new Array(nMax + 1).fill(0);
const phase = new Array(nMax + 1).fill(0);
let running = true;
let mouseIsDown = false;
let mouseClock = 0;

function colorString(hue) {
  const h = ((hue % 1) + 1) % 1;
  const i = Math.floor(h * 6);
  const f = h * 6 - i;
  const q = 1 - f;
  const rgb = [[1,f,0],[q,1,0],[0,1,f],[0,q,1],[f,0,1],[1,0,q]][i];
  return `rgb(${Math.round(rgb[0]*255)},${Math.round(rgb[1]*255)},${Math.round(rgb[2]*255)})`;
}

function init() {
  for (let n = 0; n <= nMax; n++) eigenPsi[n] = new Array(iMax + 1);
  for (let i = 0; i <= iMax; i++) {
    const x = (i - iMax / 2) / 60;
    const e0 = Math.exp(-x*x/2);
    eigenPsi[0][i] = e0;
    eigenPsi[1][i] = Math.sqrt(2)*x*e0;
    eigenPsi[2][i] = (2*x*x-1)/Math.sqrt(2)*e0;
    eigenPsi[3][i] = (2*x*x*x-3*x)/Math.sqrt(3)*e0;
    eigenPsi[4][i] = (4*x**4-12*x*x+3)/Math.sqrt(24)*e0;
    eigenPsi[5][i] = (4*x**5-20*x**3+15*x)/Math.sqrt(60)*e0;
    eigenPsi[6][i] = (8*x**6-60*x**4+90*x*x-15)/Math.sqrt(720)*e0;
    eigenPsi[7][i] = (8*x**7-84*x**5+210*x**3-105*x)/Math.sqrt(2520)*e0;
  }
  amplitude[0] = 1/Math.sqrt(2);
  amplitude[1] = 1/Math.sqrt(2);
}

function buildPsi() {
  for (let i = 0; i <= iMax; i++) { psi.re[i] = 0; psi.im[i] = 0; }
  for (let n = 0; n <= nMax; n++) {
    const re = amplitude[n] * Math.cos(phase[n]);
    const im = amplitude[n] * Math.sin(phase[n]);
    for (let i = 0; i <= iMax; i++) {
      psi.re[i] += re * eigenPsi[n][i];
      psi.im[i] += im * eigenPsi[n][i];
    }
  }
}

function paintCanvas() {
  ctx.fillStyle = 'black'; ctx.fillRect(0,0,canvas.width,canvas.height);
  const baseline = canvas.height * (1 - clockSpaceFraction);
  if (realImag.checked) {
    const y0 = baseline/2, scale = y0*0.9;
    ctx.strokeStyle='gray'; ctx.lineWidth=1; ctx.beginPath(); ctx.moveTo(0,y0); ctx.lineTo(canvas.width,y0); ctx.stroke();
    for (const key of ['re','im']) {
      ctx.beginPath(); ctx.moveTo(0,y0-psi[key][0]*scale);
      for (let i=1;i<=iMax;i++) ctx.lineTo(i,y0-psi[key][i]*scale);
      ctx.strokeStyle = key==='re' ? '#ffc000' : '#00d0ff'; ctx.lineWidth=2; ctx.stroke();
    }
  } else {
    const scale = baseline*0.55; ctx.lineWidth=2;
    for (let i=0;i<=iMax;i++) {
      ctx.beginPath(); ctx.moveTo(i,baseline);
      ctx.lineTo(i,baseline-scale*(psi.re[i]**2+psi.im[i]**2));
      let ph=Math.atan2(psi.im[i],psi.re[i]); if(ph<0) ph+=2*Math.PI;
      ctx.strokeStyle=colorString(ph/(2*Math.PI)); ctx.stroke();
    }
  }
  const space=canvas.height*clockSpaceFraction, radius=space*clockRadiusFraction;
  for(let n=0;n<=nMax;n++){
    const cx=(n+0.5)*space, cy=canvas.height-0.5*space;
    ctx.strokeStyle='gray';ctx.lineWidth=1;ctx.beginPath();ctx.arc(cx,cy,radius,0,2*Math.PI);ctx.stroke();
    ctx.beginPath();ctx.moveTo(cx,cy);ctx.lineTo(cx+radius*amplitude[n]*Math.cos(phase[n]),cy-radius*amplitude[n]*Math.sin(phase[n]));
    ctx.strokeStyle=colorString(phase[n]/(2*Math.PI));ctx.lineWidth=3;ctx.stroke();
  }
}

function frame(){
  for(let n=0;n<=nMax;n++){phase[n]-=(n+0.5)*Number(speedSlider.value); if(phase[n]<0)phase[n]+=2*Math.PI;}
  buildPsi(); paintCanvas(); if(running) setTimeout(frame,1000/30);
}

function normalize(){
  const norm=Math.sqrt(amplitude.reduce((s,a)=>s+a*a,0));
  if(norm>0) for(let n=0;n<=nMax;n++) amplitude[n]/=norm;
  buildPsi();paintCanvas();
}

function coherent(){
  const a=Number(alphaSlider.value); let fact=1;
  for(let n=0;n<=nMax;n++){if(n>0)fact*=n;amplitude[n]=Math.pow(a,n)/Math.sqrt(fact);phase[n]=0;}
  normalize();
}

function pointerPosition(e){
  const r=canvas.getBoundingClientRect();
  return {x:(e.clientX-r.left)*(canvas.width/r.width),y:(e.clientY-r.top)*(canvas.height/r.height)};
}

function setClock(p){
  const space=canvas.height*clockSpaceFraction, radius=space*clockRadiusFraction;
  mouseClock=Math.floor(p.x/space); if(mouseClock<0||mouseClock>nMax)return;
  const cx=(mouseClock+0.5)*space,cy=canvas.height-0.5*space; const dx=p.x-cx,dy=cy-p.y;
  if(dx*dx+dy*dy>radius*radius)return;
  amplitude[mouseClock]=Math.min(Math.hypot(dx,dy)/radius,1); phase[mouseClock]=Math.atan2(dy,dx); if(phase[mouseClock]<0)phase[mouseClock]+=2*Math.PI;
  buildPsi();paintCanvas();
}

canvas.addEventListener('pointerdown',e=>{mouseIsDown=true;setClock(pointerPosition(e));});
canvas.addEventListener('pointermove',e=>{if(mouseIsDown)setClock(pointerPosition(e));});
window.addEventListener('pointerup',()=>{mouseIsDown=false;});
pauseButton.onclick=()=>{running=!running;pauseButton.textContent=running?'Pause':'Resume';if(running)frame();};
document.getElementById('zeroButton').onclick=()=>{amplitude.fill(0);buildPsi();paintCanvas();};
document.getElementById('normalizeButton').onclick=normalize;
document.getElementById('coherentButton').onclick=coherent;
alphaSlider.oninput=()=>{alphaReadout.textContent=Number(alphaSlider.value).toFixed(1);};
realImag.onchange=paintCanvas;
init();buildPsi();paintCanvas();frame();
</script>
</body>
</html>
'''

display(HTML(html_template))
